# Probing AIMv2 via feature inversion

Train an identical ~20M-param decoder on frozen last-layer patch tokens of each encoder, reconstruct the input image, and compare how much pixel information survives.

- **H1 (spectrum):** AIMv2 retains more recoverable low-level detail than CLIP/SigLIP, but less than MAE.
- **H2 (prefix):** AIMv2's reconstruction error varies systematically with patch position in a way the other encoders' errors don't.

A null result on either is a valid finding.

**Running on Longleaf (UNC)**

Paths default to `/work/users/k/a/kaiji/aimv2_inversion` for outputs and caches (Hugging Face, torch, pixel cache) and `/datacommons/ImageNet/ImageNet1k` for data. The train/val folders under that root are found automatically (the Kaggle layout `ILSVRC/Data/CLS-LOC/{train,val}` with a flat val folder is supported). Run it interactively on a GPU node for milestone 1, or headless with `run_longleaf.sbatch`. Config keys can be overridden with environment variables: `FI_SMOKE=1`, `FI_TRAIN_LIMIT=10000`, `FI_EPOCHS`, `FI_ENCODERS=aimv2,clip`, `FI_OVERWRITE=1`, `FI_WORK`.

**How to use this notebook**

1. Set paths in the config cell. `smoke_test=True` runs the whole pipeline end to end on synthetic images with only the floor/ceiling "encoders" (no downloads, runs on CPU) so you can check the plumbing.
2. Milestone 1: `smoke_test=False`, `train_limit=10_000`, `encoders=["aimv2"]`. Check the recon grid looks sane and the loss drops.
3. Full run: `train_limit=None`, all encoders. Every stage is resumable: finished checkpoints and eval files are skipped unless `overwrite=True`.

**Deviations from the spec, all applied identically to every encoder**

- *Checkpoint selection* uses a separate 2k-image split drawn from ImageNet val, disjoint from the 10k eval images, so the reported numbers aren't selected on the test set.
- *Feature standardization:* tokens are standardized per channel with a fixed mean/std estimated once on training images before the decoder's `Linear(D→512)`. This is an invertible affine map, so it removes no information, but it stops encoders with very large activations (CLIP's pre-LN residual stream has outlier channels) from being handicapped by optimization. Set `standardize_features=False` to follow the spec literally.
- *Pixel cache:* instead of caching features (~26 GB per encoder), decoded 224px crops are cached once as uint8 (~7.5 GB for 50k images) and shared by every encoder. Encoders run on the fly. This also guarantees bit-identical inputs across runs.
- *Gradient clipping* at 1.0 (applied to every run).

In [ ]:
# On Longleaf, install into a conda env on /work instead (see setup_longleaf.sh). Elsewhere:
# !pip install -q "transformers>=4.50,<5" lpips torchmetrics pandas matplotlib tqdm

## 0. Config

In [ ]:
import os, json, math, time, random, gc, copy
from pathlib import Path

# Longleaf: keep caches and outputs on /work (home quota is small). Must run before transformers is imported.
WORK = Path(os.environ.get("FI_WORK", "/work/users/k/a/kaiji"))
PROJECT = WORK / "aimv2_inversion"
for _k, _sub in [("HF_HOME", "hf_cache"), ("TORCH_HOME", "torch_cache"), ("MPLCONFIGDIR", "mpl_cache")]:
    os.environ.setdefault(_k, str(WORK / _sub))
    Path(os.environ[_k]).mkdir(parents=True, exist_ok=True)

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
if os.environ.get("FI_PROGRESS", "1") == "0":   # batch jobs: no progress bars (they flood logs and stall the kernel)
    from functools import partial
    tqdm = partial(tqdm, disable=True)

CFG = dict(
    # --- paths ---
    imagenet_root="/datacommons/ImageNet/ImageNet1k",
    imagenet_train=None,         # None = auto-detect under imagenet_root (1000 class subfolders)
    imagenet_val=None,           # None = auto-detect; class subfolders OR a flat folder of 50k images
    out_dir=str(PROJECT / "results"),
    ckpt_dir=str(PROJECT / "checkpoints"),
    cache_dir=str(PROJECT / "cache"),
    # --- data ---
    train_per_class=50,          # 50k train images
    train_limit=None,            # e.g. 10_000 for milestone 1
    n_eval=10_000,               # reported metrics
    n_select=2_000,              # checkpoint selection (disjoint from eval)
    img_size=224, resize=256,
    hflip=False,                 # optional, deterministic per (seed, epoch) so all encoders see the same flips
    pixel_cache=True,
    # --- decoder / optimization (identical for every encoder) ---
    dec_dim=512, dec_depth=6, dec_heads=8, mlp_ratio=4.0,
    standardize_features=True,
    batch_size=128, epochs=30, lr=3e-4, warmup_steps=500, weight_decay=0.05,
    betas=(0.9, 0.95), grad_clip=1.0,
    compile=False,
    num_workers=max(1, min(16, int(os.environ.get("SLURM_CPUS_PER_TASK", "9")) - 1)),
    seed=0,
    encoder_init_seed=1234,      # used for the random-init AIMv2 control
    # --- which runs ---
    encoders=["aimv2", "clip", "siglip", "dinov2", "mae", "aimv2_random", "floor_zeros", "ceiling_pixels"],
    multi_seed_encoders=["aimv2", "clip"],   # extra seeds to measure decoder variance
    extra_seeds=[1, 2],
    # --- eval ---
    lpips_net="alex",            # "alex" or "vgg"; state which in the writeup
    semantic_metric=True,        # cosine sim of an independent supervised ResNet-50's features
    n_boot=1000,
    n_vis=20,
    overwrite=False,
    smoke_test=False,
)

def _env_override(cfg):
    casts = dict(FI_SMOKE=("smoke_test", lambda v: v == "1"), FI_TRAIN_LIMIT=("train_limit", int),
                 FI_EPOCHS=("epochs", int), FI_ENCODERS=("encoders", lambda v: [e.strip() for e in v.split(",") if e.strip()]),
                 FI_OVERWRITE=("overwrite", lambda v: v == "1"))
    for env, (key, cast) in casts.items():
        if os.environ.get(env, "") != "":
            cfg[key] = cast(os.environ[env]); print(f"{env} -> {key} = {cfg[key]}")
_env_override(CFG)

# keep milestone runs (train_limit) from colliding with full-run checkpoints
if CFG["train_limit"] and not CFG["smoke_test"]:
    for _k in ("out_dir", "ckpt_dir"):
        CFG[_k] = f"{CFG[_k]}_n{CFG['train_limit']}"

if CFG["smoke_test"]:
    CFG.update(train_per_class=None, n_fake_train=96, n_eval=48, n_select=16, batch_size=8, epochs=2,
               warmup_steps=4, num_workers=0, n_vis=4, n_boot=200, semantic_metric=False,
               encoders=["floor_zeros", "ceiling_pixels"], multi_seed_encoders=["ceiling_pixels"], extra_seeds=[1],
               out_dir=str(PROJECT / "results_smoke"), ckpt_dir=str(PROJECT / "checkpoints_smoke"),
               cache_dir=str(PROJECT / "cache_smoke"),
               dec_depth=2)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
USE_AMP = DEVICE == "cuda"
AMP_DTYPE = torch.bfloat16
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

OUT = Path(CFG["out_dir"]); CKPT = Path(CFG["ckpt_dir"]); CACHE = Path(CFG["cache_dir"])
EVAL_DIR, LOG_DIR, FIG_DIR = OUT / "eval", OUT / "logs", OUT / "figures"
for d in (OUT, CKPT, CACHE, EVAL_DIR, LOG_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def free_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(DEVICE, torch.__version__, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "")
print("outputs:", OUT.resolve(), "| HF cache:", os.environ["HF_HOME"], "| workers:", CFG["num_workers"])

## A. Data pipeline

Images are decoded once (resize 256 → center-crop 224) to uint8. Each batch is moved to the GPU and converted to `[0,1]`, which is the reconstruction target; each encoder wrapper applies its own mean/std on top of that. So "normalized two ways" happens per batch on the GPU from one decoded copy.

In [ ]:
IMG_EXTS = {".jpeg", ".jpg", ".png"}

def list_images(d):
    return sorted(str(p) for p in Path(d).iterdir() if p.suffix.lower() in IMG_EXTS)

class PathSource:
    def __init__(self, paths): self.items = list(paths)
    def __len__(self): return len(self.items)
    def load(self, i):
        with Image.open(self.items[i]) as im:
            return im.convert("RGB")

class FakeSource:
    """Smooth random images for the smoke test (learnable, unlike white noise)."""
    def __init__(self, n, offset): self.n, self.offset = n, offset
    def __len__(self): return self.n
    def load(self, i):
        rng = np.random.default_rng(self.offset + i)
        small = rng.integers(0, 256, (6, 6, 3), dtype=np.uint8)
        return Image.fromarray(small).resize((CFG["resize"], CFG["resize"]), Image.BICUBIC)

def find_imagenet(root, max_depth=6):
    """Find (train_dir, val_dir) under root: a folder holding train/ (1000 class dirs) and val/.
    Handles the Kaggle layout <root>/.../ILSVRC/Data/CLS-LOC/{train,val} (flat val folder)."""
    root = Path(root)
    hits, stack = [], [(root, 0)]
    while stack:
        d, depth = stack.pop()
        try:
            subdirs = [e for e in os.scandir(d) if e.is_dir()]
        except (PermissionError, FileNotFoundError, NotADirectoryError):
            continue
        names = {e.name for e in subdirs}
        if "train" in names and "val" in names:
            n_cls = sum(1 for e in os.scandir(d / "train") if e.is_dir())
            if n_cls == 1000:
                hits.append(d)
                continue
        if depth < max_depth:
            stack += [(Path(e.path), depth + 1) for e in subdirs if e.name not in {"train", "val", "test"}]
    if not hits:
        raise FileNotFoundError(f"No folder with train/ (1000 classes) and val/ under {root}. "
                                "Set CFG['imagenet_train'] and CFG['imagenet_val'] by hand.")
    hits.sort(key=lambda p: (0 if "CLS-LOC" in str(p) else 1, len(str(p))))
    if len(hits) > 1:
        print("found several ImageNet copies, using the first:", *hits, sep="\n  ")
    return str(hits[0] / "train"), str(hits[0] / "val")

def build_splits(cfg):
    path = OUT / "splits.json"
    if path.exists():
        return json.loads(path.read_text())
    rng = np.random.default_rng(cfg["seed"])
    train_root = Path(cfg["imagenet_train"])
    classes = sorted(d.name for d in train_root.iterdir() if d.is_dir())
    assert len(classes) == 1000, f"expected 1000 class folders, found {len(classes)}"
    train = []
    for c in tqdm(classes, desc="sampling train"):
        files = list_images(train_root / c)
        pick = np.sort(rng.choice(len(files), size=min(cfg["train_per_class"], len(files)), replace=False))
        train += [files[i] for i in pick]
    val_root = Path(cfg["imagenet_val"])
    subdirs = sorted(d for d in val_root.iterdir() if d.is_dir())
    val = sorted(f for d in subdirs for f in list_images(d)) if subdirs else list_images(val_root)
    assert len(val) >= cfg["n_eval"] + cfg["n_select"], f"only {len(val)} val images"
    perm = rng.permutation(len(val))
    splits = dict(
        train=train,
        eval=[val[i] for i in perm[: cfg["n_eval"]]],
        select=[val[i] for i in perm[cfg["n_eval"]: cfg["n_eval"] + cfg["n_select"]]],
    )
    path.write_text(json.dumps(splits))
    return splits

def build_sources(cfg):
    if cfg["smoke_test"]:
        return dict(train=FakeSource(cfg["n_fake_train"], 0),
                    eval=FakeSource(cfg["n_eval"], 10**6),
                    select=FakeSource(cfg["n_select"], 2 * 10**6))
    if not cfg["imagenet_train"] or not cfg["imagenet_val"]:
        cfg["imagenet_train"], cfg["imagenet_val"] = find_imagenet(cfg["imagenet_root"])
    print("train:", cfg["imagenet_train"], "\nval:  ", cfg["imagenet_val"])
    splits = build_splits(cfg)
    if cfg["train_limit"]:
        idx = np.sort(np.random.default_rng(cfg["seed"]).choice(len(splits["train"]), cfg["train_limit"], replace=False))
        splits["train"] = [splits["train"][i] for i in idx]
    return {k: PathSource(v) for k, v in splits.items()}

TRANSFORM_U8 = T.Compose([
    T.Resize(CFG["resize"], interpolation=T.InterpolationMode.BICUBIC, antialias=True),
    T.CenterCrop(CFG["img_size"]),
    T.PILToTensor(),                    # uint8 (3, 224, 224)
])

class DecodeDataset(Dataset):
    def __init__(self, source): self.source = source
    def __len__(self): return len(self.source)
    def __getitem__(self, i): return TRANSFORM_U8(self.source.load(i))

def build_pixel_cache(source, path, cfg):
    path = Path(path)
    if path.exists():
        arr = np.load(path, mmap_mode="r")
        if arr.shape[0] == len(source):
            return path
        print(f"{path} has {arr.shape[0]} images, expected {len(source)}; rebuilding")
    tmp = path.with_name(f"{path.stem}.tmp{os.getpid()}.npy")   # per-process, so two jobs can't corrupt each other
    S = cfg["img_size"]
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype=np.uint8, shape=(len(source), 3, S, S))
    dl = DataLoader(DecodeDataset(source), batch_size=256, num_workers=cfg["num_workers"])
    j = 0
    for x in tqdm(dl, desc=f"caching {path.name}"):
        arr[j: j + len(x)] = x.numpy(); j += len(x)
    arr.flush(); del arr
    os.replace(tmp, path)
    return path

class PixelDataset(Dataset):
    """Returns (uint8 image (3,S,S), index). Flips are deterministic functions of (seed, epoch, index)."""
    def __init__(self, source=None, npy_path=None, hflip=False, vflip=False, seed=0):
        assert (source is None) != (npy_path is None)
        self.source, self.npy_path = source, (str(npy_path) if npy_path else None)
        self.n = len(source) if source is not None else np.load(npy_path, mmap_mode="r").shape[0]
        self.hflip, self.vflip, self.seed = hflip, vflip, seed
        self.flip_mask, self._arr = None, None

    def set_epoch(self, epoch):
        if self.hflip:
            self.flip_mask = np.random.default_rng([self.seed, epoch]).random(self.n) < 0.5

    def __len__(self): return self.n

    def __getitem__(self, i):
        if self.npy_path is not None:
            if self._arr is None:
                self._arr = np.load(self.npy_path, mmap_mode="r")
            x = torch.from_numpy(np.ascontiguousarray(self._arr[i]))
        else:
            x = TRANSFORM_U8(self.source.load(i))
        if self.flip_mask is not None and self.flip_mask[i]:
            x = x.flip(-1)
        if self.vflip:
            x = x.flip(-2)
        return x, i

SOURCES = build_sources(CFG)
CACHE_PATHS = {}
if CFG["pixel_cache"]:
    for split, src in SOURCES.items():
        CACHE_PATHS[split] = build_pixel_cache(src, CACHE / f"{split}_{len(src)}.npy", CFG)

def make_dataset(split, hflip=False, vflip=False, seed=0):
    if split in CACHE_PATHS:
        return PixelDataset(npy_path=CACHE_PATHS[split], hflip=hflip, vflip=vflip, seed=seed)
    return PixelDataset(source=SOURCES[split], hflip=hflip, vflip=vflip, seed=seed)

def make_loader(ds, shuffle=False, seed=0, drop_last=False, batch_size=None):
    g = torch.Generator().manual_seed(seed)
    nw = CFG["num_workers"]
    return DataLoader(ds, batch_size=batch_size or CFG["batch_size"], shuffle=shuffle, generator=g,
                      drop_last=drop_last, num_workers=nw, pin_memory=(DEVICE == "cuda"),
                      persistent_workers=False)   # non-persistent so set_epoch() reaches the workers

def to_unit(x_u8):
    return x_u8.to(DEVICE, non_blocking=True).float().div_(255.0)

print({k: len(v) for k, v in SOURCES.items()})

## B. Encoder wrappers

Every wrapper maps `[0,1]` images to `(B, N, D)` **last-layer patch tokens only**: prefix tokens (CLS, registers) are dropped by keeping the last `N = grid²` tokens, and the code asserts the count. Encoders are frozen, in `eval()`, run under `no_grad`, and stored in bf16 on GPU.

| key | model | patch | grid | notes |
|---|---|---|---|---|
| `aimv2` | `apple/aimv2-large-patch14-224` | 14 | 16×16 | no CLS token |
| `clip` | `openai/clip-vit-large-patch14` | 14 | 16×16 | `last_hidden_state` is pre-`post_layernorm` |
| `siglip` | `google/siglip-large-patch16-256` | 16 | 14×14 | trained at 256px; run at 224 with interpolated pos-emb |
| `dinov2` | `facebook/dinov2-large` | 14 | 16×16 | pos-emb interpolated from 518px |
| `mae` | `facebook/vit-mae-large` | 16 | 14×14 | `mask_ratio=0` **and** identity `noise`, otherwise HF shuffles tokens |
| `aimv2_random` | AIMv2-L architecture, random init | 14 | 16×16 | architecture control |
| `floor_zeros` | all-zeros input | 14 | 16×16 | decoder can only learn the mean image |
| `ceiling_pixels` | raw 14×14 patches (588-d) | 14 | 16×16 | linear embedding of pixels |

The HF IDs are the spec's suggestions; the loaders fail loudly on missing weights, so a wrong ID or an incompatible checkpoint can't silently produce a random encoder.

In [ ]:
from transformers import AutoModel, AutoImageProcessor

CLIP_MEAN, CLIP_STD = (0.48145466, 0.4578275, 0.40821073), (0.26862954, 0.26130258, 0.27577711)
IMNET_MEAN, IMNET_STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
HALF = (0.5, 0.5, 0.5)

def patchify(x, p):
    """(B,3,H,W) -> (B, (H/p)*(W/p), p*p*3), raster (row-major) order."""
    B, C, H, W = x.shape
    g_h, g_w = H // p, W // p
    x = x.reshape(B, C, g_h, p, g_w, p).permute(0, 2, 4, 3, 5, 1)
    return x.reshape(B, g_h * g_w, p * p * C)

def unpatchify(t, p, grid):
    """(B, grid*grid, p*p*3) -> (B,3,grid*p,grid*p). Inverse of patchify."""
    B = t.shape[0]
    t = t.reshape(B, grid, grid, p, p, 3).permute(0, 5, 1, 3, 2, 4)
    return t.reshape(B, 3, grid * p, grid * p)

class Encoder(nn.Module):
    def __init__(self, name, model, call, patch_size, mean=(0, 0, 0), std=(1, 1, 1), img_size=224):
        super().__init__()
        self.name, self.model, self.call, self.patch_size = name, model, call, patch_size
        self.grid = img_size // patch_size
        self.num_tokens = self.grid ** 2
        self.register_buffer("mean", torch.tensor(mean, dtype=torch.float32).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std, dtype=torch.float32).view(1, 3, 1, 1))
        if model is not None:
            model.eval().requires_grad_(False)
        self.dim = None

    def to(self, *args, **kwargs):
        super().to(*args, **kwargs)
        if self.model is not None and DEVICE == "cuda":
            self.model.to(torch.bfloat16)
        return self

    @torch.no_grad()
    def forward(self, x01):
        x = (x01 - self.mean) / self.std
        if self.model is not None:
            x = x.to(next(self.model.parameters()).dtype)
        h = self.call(self.model, x)
        extra = h.shape[1] - self.num_tokens
        assert extra in (0, 1, 5), f"{self.name}: {h.shape[1]} tokens for a {self.grid}x{self.grid} grid"
        h = h[:, extra:].float()
        self.dim = h.shape[-1]
        return h

def _check_loading(name, info, max_missing=5):
    missing = info.get("missing_keys", [])
    if len(missing) > max_missing:
        raise RuntimeError(f"{name}: {len(missing)} missing weights, e.g. {missing[:5]}")
    if missing:
        print(f"[{name}] warning: missing keys {missing}")

def _processor_stats(hf_id, default_mean, default_std, **kw):
    try:
        p = AutoImageProcessor.from_pretrained(hf_id, **kw)
        return tuple(p.image_mean), tuple(p.image_std)
    except Exception as e:
        print(f"[{hf_id}] using default normalization ({e.__class__.__name__})")
        return default_mean, default_std

def load_aimv2(hf_id):
    """Try the native transformers class first, then the repo's remote code. Weights must load fully."""
    attempts, errors = [], []
    try:
        from transformers import Aimv2VisionModel
        attempts.append(("native", Aimv2VisionModel, {}))
    except ImportError:
        pass
    attempts.append(("remote_code", AutoModel, {"trust_remote_code": True}))
    for tag, cls, kw in attempts:
        try:
            m, info = cls.from_pretrained(hf_id, output_loading_info=True, **kw)
            _check_loading(f"aimv2/{tag}", info, max_missing=0)
            print(f"[aimv2] loaded via {tag}: {type(m).__name__}")
            return m
        except Exception as e:
            errors.append(f"{tag}: {e!r}"[:300])
    raise RuntimeError("Could not load AIMv2:\n" + "\n".join(errors))

def _aimv2_call(m, x):
    return m(pixel_values=x).last_hidden_state

def build_aimv2(random_init=False):
    hf_id = "apple/aimv2-large-patch14-224"
    m = load_aimv2(hf_id)
    if random_init:
        torch.manual_seed(CFG["encoder_init_seed"])
        ref_w = next(m.parameters()).detach().clone()
        m = type(m)._from_config(m.config)      # same class + config, fresh init
        assert not torch.equal(ref_w, next(m.parameters()).detach()), "random init equals pretrained"
    mean, std = _processor_stats(hf_id, CLIP_MEAN, CLIP_STD, trust_remote_code=True)
    return Encoder("aimv2_random" if random_init else "aimv2", m, _aimv2_call, 14, mean, std)

def build_clip():
    from transformers import CLIPVisionModel
    hf_id = "openai/clip-vit-large-patch14"
    m, info = CLIPVisionModel.from_pretrained(hf_id, output_loading_info=True); _check_loading("clip", info)
    return Encoder("clip", m, lambda m, x: m(pixel_values=x).last_hidden_state, 14, CLIP_MEAN, CLIP_STD)

def build_siglip():
    from transformers import SiglipVisionModel
    hf_id = "google/siglip-large-patch16-256"
    m, info = SiglipVisionModel.from_pretrained(hf_id, output_loading_info=True); _check_loading("siglip", info)
    call = lambda m, x: m(pixel_values=x, interpolate_pos_encoding=True).last_hidden_state
    return Encoder("siglip", m, call, 16, HALF, HALF)

def build_dinov2():
    from transformers import Dinov2Model
    hf_id = "facebook/dinov2-large"
    m, info = Dinov2Model.from_pretrained(hf_id, output_loading_info=True); _check_loading("dinov2", info)
    return Encoder("dinov2", m, lambda m, x: m(pixel_values=x).last_hidden_state, 14, IMNET_MEAN, IMNET_STD)

def build_mae():
    from transformers import ViTMAEModel
    hf_id = "facebook/vit-mae-large"
    m, info = ViTMAEModel.from_pretrained(hf_id, mask_ratio=0.0, output_loading_info=True); _check_loading("mae", info)
    def call(m, x):
        B, N = x.shape[0], (x.shape[-1] // 16) ** 2
        noise = torch.arange(N, device=x.device, dtype=torch.float32).expand(B, N)  # argsort -> identity
        out = m(pixel_values=x, noise=noise)
        assert torch.equal(out.ids_restore[0].cpu(), torch.arange(N)), "MAE token order was shuffled"
        assert int(out.mask.sum()) == 0, "MAE masked some tokens"
        return out.last_hidden_state
    return Encoder("mae", m, call, 16, IMNET_MEAN, IMNET_STD)

def build_floor():
    call = lambda m, x: x.new_zeros(x.shape[0], (x.shape[-1] // 14) ** 2, 1)
    return Encoder("floor_zeros", None, call, 14)

def build_ceiling():
    return Encoder("ceiling_pixels", None, lambda m, x: patchify(x, 14), 14, HALF, HALF)

ENCODER_BUILDERS = dict(
    aimv2=build_aimv2, aimv2_random=lambda: build_aimv2(random_init=True), clip=build_clip,
    siglip=build_siglip, dinov2=build_dinov2, mae=build_mae,
    floor_zeros=build_floor, ceiling_pixels=build_ceiling,
)

def build_encoder(name):
    return ENCODER_BUILDERS[name]().to(DEVICE)

### Token-order check

H2 depends on knowing which token sits where, so this checks it empirically instead of assuming. For a few probe cells `(r, c)` with `r ≠ c`, invert the pixels of that one patch and find the output token whose features move the most. Row-major order predicts index `r·G + c`; column-major predicts `c·G + r`. Every encoder (and the MAE un-shuffling) goes through the same test.

In [ ]:
@torch.no_grad()
def check_token_order(enc, x01, n_probe=8, seed=0):
    G, p = enc.grid, enc.patch_size
    base = enc(x01)
    rng = np.random.default_rng(seed)
    probes = [(0, G - 1), (G - 1, 0), (1, G // 2)]
    while len(probes) < n_probe:
        r, c = map(int, rng.integers(0, G, 2))
        if r != c: probes.append((r, c))
    row_major = col_major = 0
    for r, c in probes:
        x2 = x01.clone()
        x2[..., r * p:(r + 1) * p, c * p:(c + 1) * p] = 1.0 - x2[..., r * p:(r + 1) * p, c * p:(c + 1) * p]
        delta = (enc(x2) - base).norm(dim=-1).mean(0)
        k = int(delta.argmax())
        row_major += k == r * G + c
        col_major += k == c * G + r
    return dict(encoder=enc.name, grid=G, probes=len(probes), row_major_hits=row_major, col_major_hits=col_major,
                verdict="raster (row-major)" if row_major == len(probes) else "CHECK MANUALLY")

probe_x = torch.stack([make_dataset("select")[i][0] for i in range(4)])
order_rows = []
for name in CFG["encoders"]:
    if name == "floor_zeros":
        continue
    enc = build_encoder(name)
    order_rows.append(check_token_order(enc, to_unit(probe_x)))
    del enc; free_memory()
order_df = pd.DataFrame(order_rows)
order_df.to_csv(OUT / "token_order_check.csv", index=False)
order_df

## C. Decoder

`Linear(D→512)` + learned positional embedding `(N, 512)` → 6 pre-LN bidirectional transformer blocks (dim 512, 8 heads, MLP×4) → `LayerNorm` → `Linear(512 → p²·3)` → unpatchify to 224×224×3. Only the input dim, `N`, and the output head depend on the encoder.

In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, heads):
        super().__init__()
        self.heads = heads
        self.qkv = nn.Linear(dim, dim * 3)
        self.proj = nn.Linear(dim, dim)
    def forward(self, x):
        B, N, C = x.shape
        q, k, v = self.qkv(x).reshape(B, N, 3, self.heads, C // self.heads).permute(2, 0, 3, 1, 4)
        x = F.scaled_dot_product_attention(q, k, v)          # no mask: bidirectional
        return self.proj(x.transpose(1, 2).reshape(B, N, C))

class Block(nn.Module):
    def __init__(self, dim, heads, mlp_ratio):
        super().__init__()
        self.norm1, self.attn = nn.LayerNorm(dim), Attention(dim, heads)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, int(dim * mlp_ratio)), nn.GELU(), nn.Linear(int(dim * mlp_ratio), dim))
    def forward(self, x):
        x = x + self.attn(self.norm1(x))
        return x + self.mlp(self.norm2(x))

class Decoder(nn.Module):
    def __init__(self, in_dim, num_tokens, patch_size, dim=512, depth=6, heads=8, mlp_ratio=4.0):
        super().__init__()
        self.patch_size, self.grid = patch_size, int(round(num_tokens ** 0.5))
        assert self.grid ** 2 == num_tokens
        self.embed = nn.Linear(in_dim, dim)
        self.pos = nn.Parameter(torch.zeros(1, num_tokens, dim))
        self.blocks = nn.ModuleList([Block(dim, heads, mlp_ratio) for _ in range(depth)])
        self.norm = nn.LayerNorm(dim)
        self.head = nn.Linear(dim, patch_size ** 2 * 3)
        nn.init.trunc_normal_(self.pos, std=0.02)
        self.apply(self._init)

    @staticmethod
    def _init(m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)

    def forward(self, z):
        x = self.embed(z) + self.pos
        for blk in self.blocks:
            x = blk(x)
        return unpatchify(self.head(self.norm(x)), self.patch_size, self.grid)

class Standardizer(nn.Module):
    def __init__(self, mean, std):
        super().__init__()
        self.register_buffer("mean", mean.float()); self.register_buffer("std", std.float())
    def forward(self, z): return (z - self.mean) / self.std

@torch.no_grad()
def feature_stats(enc, n_images=2560):
    """Per-channel mean/std over tokens of the first n training images (same images for every encoder)."""
    ds = make_dataset("train")
    n = min(n_images, len(ds))
    loader = make_loader(torch.utils.data.Subset(ds, range(n)))
    s = ss = None; count = 0
    for x, _ in loader:
        z = enc(to_unit(x))
        z = z.reshape(-1, z.shape[-1]).double()
        s = z.sum(0) if s is None else s + z.sum(0)
        ss = (z ** 2).sum(0) if ss is None else ss + (z ** 2).sum(0)
        count += z.shape[0]
    mean = s / count
    std = (ss / count - mean ** 2).clamp_min(0).sqrt().clamp_min(1e-6)
    return mean.float(), std.float()

_d = Decoder(1024, 256, 14, CFG["dec_dim"], CFG["dec_depth"], CFG["dec_heads"], CFG["mlp_ratio"])
print(f"decoder params (D=1024, p=14): {sum(p.numel() for p in _d.parameters()) / 1e6:.1f}M")
_x = torch.rand(2, 3, 224, 224)
assert torch.equal(unpatchify(patchify(_x, 14), 14, 16), _x) and torch.equal(unpatchify(patchify(_x, 16), 16, 14), _x)
del _d, _x

## D. Training loop

Training is resumable: the full state is saved after every epoch, so a job that hits its time limit loses at most the epoch in progress, and resubmitting continues where it stopped. A run counts as finished only once `<run>.done` exists.

Plain MSE on `[0,1]` pixels. AdamW (lr 3e-4, wd 0.05 on weight matrices only), 500-step linear warmup, cosine decay, batch 128, bf16 autocast, 30 epochs (~11.7k steps at 50k images). The data order comes from a generator seeded with the run seed, so every encoder sees the same batches in the same order. The checkpoint with the best MSE on the selection split is kept.

In [ ]:
def run_name(enc_name, seed): return f"{enc_name}_s{seed}"

def lr_lambda_factory(warmup, total):
    def f(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warmup) / max(1, total - warmup))))
    return f

def param_groups(model, wd):
    decay, no_decay = [], []
    for n, p in model.named_parameters():
        (decay if p.ndim >= 2 and n != "pos" else no_decay).append(p)
    return [dict(params=decay, weight_decay=wd), dict(params=no_decay, weight_decay=0.0)]

@torch.no_grad()
def mean_mse(enc, dec, stdz, loader):
    dec.eval(); tot, n = 0.0, 0
    for x, _ in loader:
        x = to_unit(x)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = dec(stdz(enc(x)))
        tot += F.mse_loss(pred.float().clamp(0, 1), x, reduction="sum").item() / x[0].numel(); n += len(x)
    return tot / n

def _atomic_save(obj, path):
    tmp = Path(f"{path}.tmp{os.getpid()}")
    torch.save(obj, tmp); os.replace(tmp, path)

def _logged_epochs(name):
    p = LOG_DIR / f"{name}.jsonl"
    if not p.exists(): return 0
    return sum(1 for l in open(p) if '"event": "epoch"' in l)

def train_decoder(enc_name, seed, cfg=CFG):
    """Resumable at epoch granularity. Files per run in CKPT:
       <name>.pt        best checkpoint by selection-split MSE (used for eval)
       <name>_state.pt  full training state after the last finished epoch (removed when the run completes)
       <name>.done      written only after all epochs have finished"""
    name = run_name(enc_name, seed)
    ckpt_path, state_path, done_path = CKPT / f"{name}.pt", CKPT / f"{name}_state.pt", CKPT / f"{name}.done"
    log_path = LOG_DIR / f"{name}.jsonl"
    if cfg["overwrite"]:
        for p in (ckpt_path, state_path, done_path, log_path): p.unlink(missing_ok=True)
    if done_path.exists():
        print(f"[{name}] finished, skipping"); return ckpt_path
    if ckpt_path.exists() and not state_path.exists():
        # checkpoint written by an older notebook version without resume state
        n = _logged_epochs(name)
        if n >= cfg["epochs"]:
            done_path.touch(); print(f"[{name}] finished ({n} epochs logged), skipping"); return ckpt_path
        print(f"[{name}] incomplete run without resume state ({n}/{cfg['epochs']} epochs logged): restarting it")
        for p in (ckpt_path, log_path):
            if p.exists(): p.rename(p.with_name(p.name + ".incomplete"))

    enc = build_encoder(enc_name)
    _ = enc(to_unit(make_dataset("train")[0][0][None]))
    state = torch.load(state_path, map_location="cpu", weights_only=False) if state_path.exists() else None
    if state is not None:
        fmean, fstd = state["feat_mean"], state["feat_std"]
    elif cfg["standardize_features"]:
        fmean, fstd = feature_stats(enc)
    else:
        fmean, fstd = torch.zeros(enc.dim), torch.ones(enc.dim)
    stdz = Standardizer(fmean, fstd).to(DEVICE)

    set_seed(seed)
    dec = Decoder(enc.dim, enc.num_tokens, enc.patch_size, cfg["dec_dim"], cfg["dec_depth"],
                  cfg["dec_heads"], cfg["mlp_ratio"]).to(DEVICE)
    dec_fwd = torch.compile(dec) if cfg["compile"] else dec

    train_ds = make_dataset("train", hflip=cfg["hflip"], seed=seed)
    select_loader = make_loader(make_dataset("select"))
    steps_per_epoch = len(train_ds) // cfg["batch_size"]
    total = cfg["epochs"] * steps_per_epoch
    opt = torch.optim.AdamW(param_groups(dec, cfg["weight_decay"]), lr=cfg["lr"], betas=cfg["betas"])
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda_factory(cfg["warmup_steps"], total))

    best, step, start_epoch, t0 = float("inf"), 0, 0, time.time()
    if state is not None:
        dec.load_state_dict(state["decoder"]); opt.load_state_dict(state["opt"]); sched.load_state_dict(state["sched"])
        best, step, start_epoch = state["best"], state["step"], state["epoch"] + 1
        with open(log_path, "a") as f:
            f.write(json.dumps(dict(event="resume", from_epoch=start_epoch, step=step)) + "\n")
        print(f"[{name}] resuming at epoch {start_epoch} (best select MSE so far {best:.5f})", flush=True)
    else:
        log_path.write_text(json.dumps(dict(event="config", run=name, encoder=enc_name, seed=seed, in_dim=enc.dim,
                                            num_tokens=enc.num_tokens, patch=enc.patch_size, total_steps=total,
                                            cfg={k: v for k, v in cfg.items() if k != "encoders"})) + "\n")

    for epoch in range(start_epoch, cfg["epochs"]):
        # batch order depends only on (seed, epoch): identical across encoders and across resumes
        train_ds.set_epoch(epoch); dec.train()
        train_loader = make_loader(train_ds, shuffle=True, seed=seed * 100_003 + epoch, drop_last=True)
        run_loss, te = 0.0, time.time()
        for x, _ in tqdm(train_loader, desc=f"{name} ep{epoch}", leave=False):
            x = to_unit(x)
            z = stdz(enc(x))
            with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
                pred = dec_fwd(z)
            loss = F.mse_loss(pred.float(), x)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            if cfg["grad_clip"]:
                torch.nn.utils.clip_grad_norm_(dec.parameters(), cfg["grad_clip"])
            opt.step(); sched.step(); step += 1
            run_loss += loss.item()
            if step % 50 == 0:
                with open(log_path, "a") as f:
                    f.write(json.dumps(dict(event="train", step=step, loss=loss.item(), lr=sched.get_last_lr()[0],
                                            sec_per_step=(time.time() - te) / (step - epoch * steps_per_epoch))) + "\n")
        val = mean_mse(enc, dec_fwd, stdz, select_loader)
        rec = dict(event="epoch", epoch=epoch, step=step, train_loss=run_loss / len(train_loader), select_mse=val,
                   epoch_minutes=(time.time() - te) / 60, minutes=(time.time() - t0) / 60)
        with open(log_path, "a") as f: f.write(json.dumps(rec) + "\n")
        print(f"[{name}] epoch {epoch}: train {rec['train_loss']:.5f}  select {val:.5f}  "
              f"({rec['epoch_minutes']:.1f} min/epoch)", flush=True)
        if val < best:
            best = val
            _atomic_save(dict(decoder=dec.state_dict(), feat_mean=fmean, feat_std=fstd, encoder=enc_name, seed=seed,
                              in_dim=enc.dim, num_tokens=enc.num_tokens, patch=enc.patch_size, epoch=epoch,
                              select_mse=val, cfg=cfg), ckpt_path)
        _atomic_save(dict(decoder=dec.state_dict(), opt=opt.state_dict(), sched=sched.state_dict(), epoch=epoch,
                          step=step, best=best, feat_mean=fmean, feat_std=fstd), state_path)
    done_path.touch(); state_path.unlink(missing_ok=True)
    print(f"[{name}] done in {(time.time() - t0) / 60:.1f} min", flush=True)
    del enc, dec, dec_fwd, opt; free_memory()
    return ckpt_path

def load_run(name):
    ck = torch.load(CKPT / f"{name}.pt", map_location="cpu", weights_only=False)
    c = ck["cfg"]
    enc = build_encoder(ck["encoder"])
    dec = Decoder(ck["in_dim"], ck["num_tokens"], ck["patch"], c["dec_dim"], c["dec_depth"], c["dec_heads"], c["mlp_ratio"])
    dec.load_state_dict(ck["decoder"]); dec.to(DEVICE).eval()
    return enc, dec, Standardizer(ck["feat_mean"], ck["feat_std"]).to(DEVICE), ck

In [ ]:
RUNS = [(e, CFG["seed"]) for e in CFG["encoders"]]
RUNS += [(e, s) for e in CFG["multi_seed_encoders"] for s in CFG["extra_seeds"] if e in CFG["encoders"]]
for enc_name, seed in RUNS:
    train_decoder(enc_name, seed)
missing = [run_name(e, s) for e, s in RUNS if not (CKPT / f"{run_name(e, s)}.done").exists()]
assert not missing, f"unfinished runs: {missing}"

In [ ]:
def plot_curves(names):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for n in names:
        rows = [json.loads(l) for l in open(LOG_DIR / f"{n}.jsonl")]
        tr = [r for r in rows if r["event"] == "train"]; ep = [r for r in rows if r["event"] == "epoch"]
        if tr: axes[0].plot([r["step"] for r in tr], [r["loss"] for r in tr], label=n, lw=0.8)
        axes[1].plot([r["epoch"] for r in ep], [r["select_mse"] for r in ep], label=n, marker=".")
    for ax, t in zip(axes, ["train MSE", "selection-split MSE"]):
        ax.set_title(t); ax.set_yscale("log"); ax.legend(fontsize=7)
    plt.tight_layout(); plt.savefig(FIG_DIR / "training_curves.png", dpi=120); plt.show()

plot_curves([run_name(e, s) for e, s in RUNS])

## E. Evaluation

Per-image MSE, PSNR, SSIM (11×11 Gaussian, σ=1.5) and LPIPS on reconstructions clamped to `[0,1]`, averaged with bootstrap 95% CIs over eval images. The optional semantic metric is the cosine similarity of a supervised ImageNet ResNet-50's pooled features (torchvision), a model that is not one of the compared encoders.

Each eval pass also stores per-image error maps on a common 16×16 grid of 14px cells (used for all cross-encoder position comparisons, including the 14×14-token MAE and SigLIP) and on each encoder's native token grid, plus reconstructions of the first `n_vis` eval images.

In [ ]:
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn

_LPIPS = None
def get_lpips():
    global _LPIPS
    if _LPIPS is None:
        import lpips
        _LPIPS = lpips.LPIPS(net=CFG["lpips_net"], verbose=False).to(DEVICE).eval()
    return _LPIPS

_SEM = None
def get_semantic():
    global _SEM
    if _SEM is None:
        import torchvision
        m = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V2)
        m.fc = nn.Identity()
        _SEM = m.to(DEVICE).eval()
    return _SEM

@torch.no_grad()
def semantic_sim(a, b):
    m = get_semantic()
    mean = torch.tensor(IMNET_MEAN, device=DEVICE).view(1, 3, 1, 1); std = torch.tensor(IMNET_STD, device=DEVICE).view(1, 3, 1, 1)
    fa, fb = m((a - mean) / std), m((b - mean) / std)
    return F.cosine_similarity(fa, fb, dim=-1)

COMMON_CELL = 14   # 16x16 common grid

@torch.no_grad()
def evaluate_run(name, vflip=False, cfg=CFG):
    path = EVAL_DIR / f"{name}{'_vflip' if vflip else ''}.npz"
    if path.exists() and not cfg["overwrite"]:
        return dict(np.load(path))
    enc, dec, stdz, ck = load_run(name)
    loader = make_loader(make_dataset("eval", vflip=vflip), batch_size=64)
    lp = get_lpips()
    keys = ["mse", "psnr", "ssim", "lpips"] + (["sem"] if cfg["semantic_metric"] else [])
    acc = {k: [] for k in keys}; common, native, recon = [], [], []
    pix_sum = torch.zeros(cfg["img_size"], cfg["img_size"], device=DEVICE, dtype=torch.float64)
    for x, idx in tqdm(loader, desc=f"eval {path.stem}", leave=False):
        x = to_unit(x)
        with torch.autocast(DEVICE, dtype=AMP_DTYPE, enabled=USE_AMP):
            pred = dec(stdz(enc(x)))
        pred = pred.float().clamp(0, 1)
        err = ((pred - x) ** 2).mean(1)                                   # (B,H,W)
        mse = err.mean((1, 2))
        acc["mse"].append(mse); acc["psnr"].append(10 * torch.log10(1.0 / mse.clamp_min(1e-10)))
        acc["ssim"].append(ssim_fn(pred, x, data_range=1.0, reduction="none"))
        acc["lpips"].append(lp(pred * 2 - 1, x * 2 - 1).flatten())
        if cfg["semantic_metric"]:
            acc["sem"].append(semantic_sim(pred, x))
        common.append(F.avg_pool2d(err[:, None], COMMON_CELL)[:, 0].half())
        native.append(F.avg_pool2d(err[:, None], enc.patch_size)[:, 0].half())
        pix_sum += err.double().sum(0)
        keep = idx < cfg["n_vis"]
        if keep.any():
            recon.append((pred[keep.to(DEVICE)] * 255).round().byte().cpu())
    res = {k: torch.cat(v).float().cpu().numpy() for k, v in acc.items()}
    res["cells_common"] = torch.cat(common).float().cpu().numpy()
    res["cells_native"] = torch.cat(native).float().cpu().numpy()
    res["pixel_map"] = (pix_sum / len(loader.dataset)).float().cpu().numpy()
    res["recon"] = torch.cat(recon).numpy() if recon else np.zeros((0, 3, 224, 224), np.uint8)
    np.savez_compressed(path, **res)
    del enc, dec; free_memory()
    return res

EVALS, EVALS_FLIP = {}, {}
for enc_name, seed in RUNS:
    n = run_name(enc_name, seed)
    EVALS[n] = evaluate_run(n)
    if seed == CFG["seed"]:
        EVALS_FLIP[n] = evaluate_run(n, vflip=True)   # flip control (no retraining)

In [ ]:
METRICS = ["mse", "psnr", "ssim", "lpips"] + (["sem"] if CFG["semantic_metric"] else [])
HIGHER_BETTER = dict(mse=False, psnr=True, ssim=True, lpips=False, sem=True)

def bootstrap_mean_ci(v, n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), (n_boot, len(v)))
    means = v[idx].mean(1)
    return v.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

def fmt(m, lo, hi, nd=4): return f"{m:.{nd}f} [{lo:.{nd}f}, {hi:.{nd}f}]"

main_runs = [run_name(e, CFG["seed"]) for e in CFG["encoders"]]
rows = []
for n in main_runs:
    r = dict(run=n)
    for k in METRICS:
        m, lo, hi = bootstrap_mean_ci(EVALS[n][k])
        r[k], r[f"{k}_lo"], r[f"{k}_hi"] = m, lo, hi
    rows.append(r)
metrics_df = pd.DataFrame(rows).sort_values("mse")
metrics_df.to_csv(OUT / "metrics.csv", index=False)

pretty = pd.DataFrame({"run": metrics_df["run"]})
for k in METRICS:
    nd = 2 if k == "psnr" else 4
    pretty[k] = [fmt(r[k], r[f"{k}_lo"], r[f"{k}_hi"], nd) for _, r in metrics_df.iterrows()]
print(f"LPIPS net: {CFG['lpips_net']}   |   mean [95% bootstrap CI] over {len(EVALS[main_runs[0]]['mse'])} eval images")
pretty.reset_index(drop=True)

Paired bootstrap of `AIMv2 − other` for each metric (same images resampled for both), which is the direct test for H1. Positive PSNR/SSIM/sem differences and negative MSE/LPIPS differences mean AIMv2 is better.

In [ ]:
def paired_diff_ci(a, b, n_boot=None, seed=0):
    n_boot = n_boot or CFG["n_boot"]
    d = a - b
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), (n_boot, len(d)))].mean(1)
    return d.mean(), np.percentile(means, 2.5), np.percentile(means, 97.5)

REF = run_name("aimv2", CFG["seed"])
diff_rows = []
if REF in EVALS:
    for n in main_runs:
        if n == REF: continue
        r = dict(other=n)
        for k in METRICS:
            m, lo, hi = paired_diff_ci(EVALS[REF][k], EVALS[n][k])
            better = (lo > 0 if HIGHER_BETTER[k] else hi < 0)
            worse = (hi < 0 if HIGHER_BETTER[k] else lo > 0)
            r[k] = fmt(m, lo, hi, 2 if k == "psnr" else 4) + ("  AIMv2 better" if better else "  AIMv2 worse" if worse else "  n.s.")
        diff_rows.append(r)
diff_df = pd.DataFrame(diff_rows)
diff_df.to_csv(OUT / "aimv2_paired_diffs.csv", index=False)
diff_df

Decoder variance across seeds (mean of per-image metric for each seed, then mean ± std across seeds).

In [ ]:
seed_rows = []
for e in CFG["multi_seed_encoders"]:
    names = [run_name(e, s) for s in [CFG["seed"]] + CFG["extra_seeds"] if run_name(e, s) in EVALS]
    if len(names) < 2: continue
    r = dict(encoder=e, n_seeds=len(names))
    for k in METRICS:
        vals = np.array([EVALS[n][k].mean() for n in names])
        r[k] = f"{vals.mean():.4f} ± {vals.std(ddof=1):.4f}"
    seed_rows.append(r)
pd.DataFrame(seed_rows)

## F. Position analysis (H2)

For each encoder, average the per-cell error over eval images, then divide by the map's own mean so encoders with different overall error are comparable. Profiles are plotted against row, column, and raster index. Slopes are the change in normalized error from the first to the last position (linear fit on index scaled to `[0,1]`), with bootstrap CIs over images. AIMv2 vs. others uses a paired bootstrap of slope differences and the correlation between normalized maps.

In [ ]:
def norm_map(M): return M / M.mean()

def slopes(M):
    """Linear-fit slope (first->last) of a normalized GxG map vs row, column, raster index."""
    G = M.shape[0]
    def s(y):
        t = np.linspace(0, 1, len(y)); return np.polyfit(t, y, 1)[0]
    return dict(row=s(M.mean(1)), col=s(M.mean(0)), raster=s(M.reshape(-1)))

def bootstrap_slopes(C, n_boot=None, seed=0, idx=None):
    n_boot = n_boot or CFG["n_boot"]
    if idx is None:
        idx = np.random.default_rng(seed).integers(0, len(C), (n_boot, len(C)))
    out = {k: [] for k in ("row", "col", "raster")}
    for ii in idx:
        for k, v in slopes(norm_map(C[ii].mean(0))).items(): out[k].append(v)
    return {k: np.array(v) for k, v in out.items()}

maps = {n: norm_map(EVALS[n]["cells_common"].mean(0)) for n in main_runs}

# slopes with CIs, plus paired differences vs AIMv2 using shared resample indices
n_img = len(EVALS[main_runs[0]]["cells_common"])
boot_idx = np.random.default_rng(0).integers(0, n_img, (CFG["n_boot"], n_img))
boots = {n: bootstrap_slopes(EVALS[n]["cells_common"].astype(np.float32), idx=boot_idx) for n in main_runs}

pos_rows = []
for n in main_runs:
    point = slopes(maps[n]); r = dict(run=n)
    for k in ("row", "col", "raster"):
        lo, hi = np.percentile(boots[n][k], [2.5, 97.5]); r[f"{k}_slope"] = fmt(point[k], lo, hi, 3)
        if REF in boots and n != REF:
            d = boots[REF][k] - boots[n][k]; dlo, dhi = np.percentile(d, [2.5, 97.5])
            r[f"aimv2_minus_this_{k}"] = fmt(slopes(maps[REF])[k] - point[k], dlo, dhi, 3)
    if REF in maps and n != REF:
        r["corr_with_aimv2_map"] = np.corrcoef(maps[REF].ravel(), maps[n].ravel())[0, 1]
    pos_rows.append(r)
pos_df = pd.DataFrame(pos_rows)
pos_df.to_csv(OUT / "position_slopes.csv", index=False)
pos_df

In [ ]:
def plot_heatmaps(maps, title, fname):
    names = list(maps)
    vmin = min(m.min() for m in maps.values()); vmax = max(m.max() for m in maps.values())
    cols = min(4, len(names)); rows_ = math.ceil(len(names) / cols)
    fig, axes = plt.subplots(rows_, cols, figsize=(3.2 * cols, 3.2 * rows_), squeeze=False)
    for ax in axes.ravel(): ax.axis("off")
    for ax, n in zip(axes.ravel(), names):
        im = ax.imshow(maps[n], cmap="magma", vmin=vmin, vmax=vmax); ax.set_title(n, fontsize=9)
    fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, label="error / mean error")
    fig.suptitle(title); plt.savefig(FIG_DIR / fname, dpi=130, bbox_inches="tight"); plt.show()

plot_heatmaps(maps, "Normalized per-cell MSE (common 16×16 grid)", "error_heatmaps_common.png")
plot_heatmaps({n: norm_map(EVALS[n]["cells_native"].mean(0)) for n in main_runs},
              "Normalized per-token MSE (native token grid)", "error_heatmaps_native.png")

fig, axes = plt.subplots(1, 3, figsize=(17, 4))
for n in main_runs:
    M = maps[n]; lw = 2.2 if n == REF else 1.0
    axes[0].plot(M.mean(1), label=n, lw=lw); axes[1].plot(M.mean(0), label=n, lw=lw)
    rast = M.reshape(-1); k = 16
    axes[2].plot(np.convolve(rast, np.ones(k) / k, mode="valid"), label=n, lw=lw)
for ax, t in zip(axes, ["vs row index", "vs column index", "vs raster index (16-cell moving avg)"]):
    ax.set_title(t); ax.set_ylabel("error / mean"); ax.axhline(1, c="gray", lw=0.5)
axes[0].legend(fontsize=7)
plt.tight_layout(); plt.savefig(FIG_DIR / "position_profiles.png", dpi=130); plt.show()

if REF in maps:
    others = [n for n in main_runs if n != REF and not n.startswith(("floor", "ceiling", "aimv2_random"))]
    if others:
        diff = maps[REF] - np.mean([maps[n] for n in others], 0)
        v = np.abs(diff).max()
        plt.figure(figsize=(4, 4)); plt.imshow(diff, cmap="RdBu_r", vmin=-v, vmax=v); plt.colorbar(shrink=0.8)
        plt.title("AIMv2 − mean(other pretrained)\nnormalized error"); plt.axis("off")
        plt.savefig(FIG_DIR / "aimv2_minus_others.png", dpi=130, bbox_inches="tight"); plt.show()

### Flip control

Eval images are flipped vertically (input and target), with no retraining. Maps stay in token/image coordinates. If the error pattern is tied to token position, the flipped-eval map should match the original map (`corr_token` high, row slope keeps its sign). If it's driven by content (sky at the top), it should match the vertically flipped original (`corr_content` high, row slope flips sign). `self_symmetry` is the correlation of the original map with its own vertical flip: when it's high, the test can't tell the two apart.

In [ ]:
flip_rows = []
fig, axes = plt.subplots(1, len(EVALS_FLIP), figsize=(3.3 * len(EVALS_FLIP), 3.2), squeeze=False)
for ax, n in zip(axes[0], EVALS_FLIP):
    E0 = maps[n]; E1 = norm_map(EVALS_FLIP[n]["cells_common"].mean(0))
    c = lambda a, b: np.corrcoef(a.ravel(), b.ravel())[0, 1]
    flip_rows.append(dict(run=n, corr_token=c(E1, E0), corr_content=c(E1, np.flipud(E0)),
                          self_symmetry=c(E0, np.flipud(E0)),
                          row_slope_orig=slopes(E0)["row"], row_slope_vflip=slopes(E1)["row"],
                          raster_slope_orig=slopes(E0)["raster"], raster_slope_vflip=slopes(E1)["raster"]))
    ax.plot(E0.mean(1), label="original"); ax.plot(E1.mean(1), label="v-flipped eval")
    ax.plot(np.flipud(E0).mean(1), "--", label="original, flipped (content pred.)", lw=0.8)
    ax.set_title(n, fontsize=8); ax.set_xlabel("row (token coords)")
axes[0, 0].legend(fontsize=6)
plt.tight_layout(); plt.savefig(FIG_DIR / "flip_control_rows.png", dpi=130); plt.show()

flip_df = pd.DataFrame(flip_rows)
flip_df["reading"] = np.where(flip_df.self_symmetry > 0.8, "map ~symmetric: uninformative",
                     np.where(flip_df.corr_token > flip_df.corr_content, "position-tied", "content-tied"))
flip_df.to_csv(OUT / "flip_control.csv", index=False)
flip_df.round(3)

## G. Visualization

In [ ]:
def recon_grid(names, n_show=None, fname="recon_grid.png"):
    n_show = n_show or CFG["n_vis"]
    ds = make_dataset("eval")
    orig = torch.stack([ds[i][0] for i in range(n_show)]).permute(0, 2, 3, 1).numpy()
    cols = ["original"] + names
    fig, axes = plt.subplots(n_show, len(cols), figsize=(1.6 * len(cols), 1.6 * n_show), squeeze=False)
    for i in range(n_show):
        for j, col in enumerate(cols):
            img = orig[i] if col == "original" else EVALS[col]["recon"][i].transpose(1, 2, 0)
            axes[i, j].imshow(img); axes[i, j].set_xticks([]); axes[i, j].set_yticks([])
            if i == 0: axes[i, j].set_title(col.replace("_s0", ""), fontsize=7)
            if j > 0: axes[i, j].set_xlabel(f"{EVALS[col]['psnr'][i]:.1f} dB", fontsize=6)
    plt.tight_layout(); plt.savefig(FIG_DIR / fname, dpi=110); plt.show()

recon_grid(main_runs)

## Summary: H1 and H2

In [ ]:
def h1_summary():
    need = [run_name(e, CFG["seed"]) for e in ("aimv2", "clip", "mae")]
    if not all(n in EVALS for n in need):
        return "H1: missing runs, can't evaluate."
    lines = []
    for k in ("mse", "lpips", "psnr", "ssim"):
        verdicts = {}
        for other in ("clip", "siglip", "mae"):
            n = run_name(other, CFG["seed"])
            if n not in EVALS: continue
            m, lo, hi = paired_diff_ci(EVALS[REF][k], EVALS[n][k])
            better = lo > 0 if HIGHER_BETTER[k] else hi < 0
            worse = hi < 0 if HIGHER_BETTER[k] else lo > 0
            verdicts[other] = "better" if better else "worse" if worse else "n.s."
        held = verdicts.get("clip") == "better" and verdicts.get("siglip", "better") == "better" and verdicts.get("mae") == "worse"
        lines.append(f"  {k:6s} AIMv2 vs {verdicts}  ->  {'consistent with H1' if held else 'NOT consistent with H1'}")
    return "H1 (CLIP/SigLIP < AIMv2 < MAE):\n" + "\n".join(lines)

def h2_summary():
    if REF not in boots: return "H2: no AIMv2 run."
    lines = []
    for k in ("row", "col", "raster"):
        lo, hi = np.percentile(boots[REF][k], [2.5, 97.5])
        diffs = []
        for n in main_runs:
            if n == REF or n.startswith(("floor", "ceiling")): continue
            d = boots[REF][k] - boots[n][k]; dlo, dhi = np.percentile(d, [2.5, 97.5])
            diffs.append(f"{n.replace('_s0','')}: {'≠' if (dlo > 0 or dhi < 0) else '≈'}")
        lines.append(f"  {k:6s} AIMv2 slope {slopes(maps[REF])[k]:+.3f} [{lo:+.3f}, {hi:+.3f}];  vs others: {', '.join(diffs)}")
    if REF in EVALS_FLIP:
        fr = flip_df.set_index("run").loc[REF]
        lines.append(f"  flip control: corr_token={fr.corr_token:.2f}, corr_content={fr.corr_content:.2f}, "
                     f"self_symmetry={fr.self_symmetry:.2f} -> {fr.reading}")
    return "H2 (AIMv2-specific positional error pattern):\n" + "\n".join(lines)

print(h1_summary()); print(); print(h2_summary())

### Writeup notes

Fill in after looking at the numbers and figures above.

**H1.** Where does AIMv2 sit between CLIP/SigLIP and MAE on each metric? Do MSE and LPIPS agree? How far is each encoder from the floor (mean image) and the ceiling (raw pixels)? Is the random-init AIMv2 close to the trained one (if so, pretrained features add little beyond the architecture)?

**H2.** Is there a positional gradient in AIMv2's normalized error map that the other encoders don't show (slope differences with CIs excluding 0)? Does it survive the flip control? Is the random-init AIMv2 different from the trained one in this respect?

**Caveats to state.**
- The released AIMv2 encoder runs bidirectionally at inference (equivalent to `M = I`), so early patches see the whole image; a null H2 is plausible and legitimate.
- Raster position is confounded with content; the flip control only separates them where the error map isn't vertically symmetric.
- Encoders differ in native resolution and patch size (SigLIP trained at 256px; MAE/SigLIP use 16px patches, so 196 tokens vs. 256). Cross-encoder position comparisons use a common 16×16 pixel grid.
- The decoder is one fixed capacity and objective (MSE); a stronger decoder could recover more from every encoder. Seed variance is reported for the multi-seed encoders.
- Features are from the last layer only; CLIP's `last_hidden_state` is before its final `post_layernorm`, while AIMv2/DINOv2/MAE/SigLIP include their final norm. With per-channel standardization this difference is largely an affine one.
- LPIPS network used: see the config (`lpips_net`).